**FEBS PRACTICAL COURSE 2026: Generative AI for protein engineering**

**Notebook developed by Núria Mimbrero.**

# *****In silico*** filtering of protein designs 🔎**
* Once we have generated candidate sequences using a Protein Language Model (PLM), the next critical step is filtering.


* Autoregressive PLMs have the potential to generate millions of proteins in short timeframes. This creates a new challenge when it comes to selecting protein candidates for experimental characterization. Robust filtering pipelines of candidates are thus necessary to maximize the efficiency of experimental efforts.  Experimental validation is expensive and slow (e.g., cloning, expression, purification, assays), so we want to **prioritize only the most promising designs**. A good filtering pipeline can dramatically increase success rates while reducing cost and time.

* Goal: **Maximize the probability that tested designs are functional and well-folded**.

* Multiple prediction tools exist for each protein property of interest, with varying degrees of accuracy and applicability to proteins distant from the training data

* Which properties are suitable for filtering generated protein candidates **depends on the intended activity** the protein shall have. For example, designing an enzyme requires different filtering steps than designing a binder, because different in silico metrics can be predictive for experimental success.

We do this by combining sequence-based, structure-based, and biophysical filters.


## 🔤**SEQUENCE-BASED FILTERS :**

:Evaluate whether the sequence "looks like" a natural, functional protein.

### **1. Perplexity**
How "surpising" is a sequence to a protein language model.
PLMs already reeport their cown confidence metric in the form of perplexity of the generated sequence.

Perplexity: exponentiated average negative log-likelihood of a generated sequence, which highly depends on the input (control tag) and the trained weights. You can find the mathematical intuition here: https://huggingface.co/docs/transformers/perplexity
- Low perplexity -> sequence resembles natural proteins
- High perplexity -> sequence that goes far from the natural distribution (out-of-distribution).

**Interpretation**:
* **A lower perplexity corresponds to a higher likelihood of the sequence and a higher model-internal confidence**

### **2. ESM-1v (Variant Effect Prediction)**

Estimates how **likely mutations are compared to a reference wt sequence.**
This metric is useful when designing variants of known proteins to help detecting deleterious mutations.
* Low ESM-1v value (in absolute value) -> potentially disruptive
* High ESM-1v value (in absolute value)  -> more tolerated mutations

**Interpretation:**
* **Lower score indicate good designs**

### **3. EC Label Prediction (CLEAN)**
Predicts the enzimatic function (EC number from a sequence).
* Ensures the designs retain their **intended function**

**Interpretation:**
* Keep sequences with correct EC class prediction


### **4. Sequence Similarity (MMseqs or BLAST)**
Computes the novelty of the deisgn in respect to natural enzymes.

**Interpretation:**
* Discard 100% seq id to ensure the sequence is new
Note: you can play with this score depending on how novel you want your sequence to be.

## 🚵 **STRUCTURE-BASED FILTERS**

Evaluate if the sequences are likely to **fold correctly**.

### **4. AlphaFold/ESMFold pLDDT**
Predict the structure + confidence.
* pLDDT: local confidence metric that estimates how well a predicted structure would match an experimentally determined structure of the same protein.

**Interpretation**:
plddt ranges from 0 to 100
* greater than 90   → very high confidence
* 70–90 → good
* lower than 70  → unreliable regions

You can compute ESMfold online here (up to 400 amino acids): https://esmatlas.com/

### **5. TM-score (Fold similarity)**

Compare predicted structure to a Wild-type (WT) or Target scaffold (using Foldseek)

**Interpretation:**
* higher than 0.7 → same fold
* 0.5–0.7 → partial similarity
* lower than 0.5 → different fold

Goal: Preserve fold while allowing sequence diversity

## ✅   **SEQUENCE-STUCTURE CONSISTENCY**

### **6. ProteinMPNN Score**
Tests inverse folding consistency, given a structure, how compatible is the sequence?

**Interpretation:**
* Lower score (in absolute value)  → sequence fits structure well
* Higher score (in absolute value) → sequence/structure mismatch

 ## 🧬 **BIOPHYSICAL AND ROSETTA-BASED FILTERS**

### **7. SASA (Solvent Accessible Surface Area)**

Measures exposed surface area. It is related to folding and solubility

**Interpretation:**
* Lower -> better
* Filter by relative_hydrophobic_sasa <3:

### **8. SAP (Aggregation Propensity)**

Detect hydrophobic patches that may aggregate


**Interpretation:**

* Lower SAP → better (less aggregation risk)
* filter by averaged_sap <= 0.4ç

note: average_sap is computed by diving the SAP score by the lenght of the sequence


### **9. Charge**

Net charge affects solubility and stability

**Interpretation**:
* Avoid extreme charge and net charge
note that

Note: to compute the net charge of a protein sequence, we count it according to charged amino acids.
*     +1: K (Lys), R (Arg), H
*     -1: D (Asp), E (Glu)

## **🧯CATALYTIC RESIDUES**
For the cases where you target enzyme is well documented and you have the knowledge of the key residues that enable catalysis, it is very important for you to make sure they are present in your selected designs.

There are different ways to check this. For instance: checking the sequence trought the command line and structurally superpose the structure with the known wt in pymol.

# **HAND ON PRACTICE:**

Imagine you computed all the in silico metrics and you obtain the values in the following table. Follow the tutorial to make sure you chose the best **3 candidates**.

**1. Download excel file with the results**

In [ ]:
!pip -q install -U gdown
!gdown --folder "https://drive.google.com/drive/folders/1lvXneaK03-p5Yb_vDV1XNUJV18tjKUM1?usp=sharing" -O candidates

You can find the file inside candidates folder in data

**2. Load the file**

In [ ]:
import os
os.getcwd()

In [ ]:
import pandas as pd

file_path = "/content/candidates/candidates.xlsx"
df = pd.read_excel(file_path)

df.head()

In [ ]:
# Columns that should be numeric, now the decimals are writen as ,
numeric_cols = [
    "perplexity", "esm1v_score", "sequence_similarity",
    "plddt", "tm-score", "proteinmpnn_score",
    "relative_hydrophobic_sasa", "average_sap", "charge"
]

# Replace comma with dot and convert
for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(",", ".").astype(float)

df.dtypes

In [ ]:
df.head()

**3. Filter by**
* perplexity <= 75th percentile, if you generate more sequences you can lower this value to 25 for example
* plddt > 70
* tm-score > 0.8
* sequence_similarity != 1.00
* average_sap <= 0.4
* relative_hydrophobic_sasa < 3
* charge != 0
* EC label = 3.8.1.- (dehalogenases in this case )
* rank by: esm1v_score + proteinmpnn_score (lowest absolute value)


In [ ]:
import numpy as np

# 75th percentile cutoff for perplexity
ppx_cutoff = df["perplexity"].quantile(0.75)

filtered = df[
    (df["perplexity"] <= ppx_cutoff) &
    (df["plddt"] > 70) &
    (df["tm-score"] > 0.8) &
    (df["sequence_similarity"] != 1.00) &
    (df["average_sap"] <= 0.4) &
    (df["relative_hydrophobic_sasa"] < 3) &
    (df["charge"] != 0) &
    (df["clean_prediction"] == "EC:3.8.1.3")
].copy()

print("Remaining candidates after filtering:", len(filtered))

In [ ]:
filtered["esm1v_abs"] = filtered["esm1v_score"].abs()
filtered["mpnn_abs"] = filtered["proteinmpnn_score"].abs()

filtered["final_score"] = (
    filtered["esm1v_abs"].rank() +
    filtered["mpnn_abs"].rank()
)


top3 = filtered.sort_values("final_score").head(3)

top3